# 🧱 Arquitetura Medallion — Camada **Bronze** (Ingestão)

Este notebook organiza o pipeline do Titanic em camadas (padrão *medallion*):

| Camada | Papel | Saída |
|--------|-------|-------|
| **Bronze** | **Ingestão** do dado **cru** da fonte, *sem transformação*, com **metadados de controle** (linhagem/auditoria) e **idempotência** | `camada_bronze/titanic.csv` |
| Prata (silver) | Limpeza e tratamento (valores ausentes, tipos) | *em memória* |
| **Ouro (gold)** | Features prontas para análise/modelo | `camada_gold/titanic_tratado.csv` |

**O que a camada Bronze garante:**
- **Fidelidade:** copia o dado exatamente como veio da fonte (nenhuma regra de negócio aqui).
- **Metadados de controle:** colunas com prefixo `_` (linhagem) + um *sidecar* `_bronze_metadata.json` (auditoria do lote).
- **Idempotência:** a chave é o `sha256` do conteúdo da fonte. **Rodar 2× = mesmo resultado** (arquivo byte-a-byte idêntico); se a fonte não mudou, a ingestão nem reprocessa.


### 1) Camada **Bronze** — ingestão do dado cru (metadados de controle + idempotência)


In [3]:
!pip install pandas numpy


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# ── CAMADA BRONZE (INGESTÃO) ─────────────────────────────────────────────────
# Traz o dado CRU da fonte para o projeto SEM transformar, adicionando apenas
# METADADOS DE CONTROLE (linhagem) e garantindo IDEMPOTÊNCIA (rodar 2x = mesmo resultado).
import os, io, json, hashlib
from datetime import datetime, timezone
import pandas as pd
import numpy as np  # usado nas próximas camadas

URL_FONTE   = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
BRONZE_CSV  = "camada_bronze/titanic.csv"
BRONZE_META = "camada_bronze/_bronze_metadata.json"


def _ler_fonte(fonte):
    """Lê os bytes crus da fonte (URL http(s) ou caminho local)."""
    if str(fonte).startswith(("http://", "https://")):
        import urllib.request
        with urllib.request.urlopen(fonte, timeout=30) as r:
            return r.read()
    with open(fonte, "rb") as f:
        return f.read()


def ingerir_camada_bronze(fonte=URL_FONTE, destino=BRONZE_CSV, meta_path=BRONZE_META):
    """Ingestão da camada BRONZE do Titanic.

    - Extrai o CSV cru da `fonte` (não altera os dados).
    - Adiciona METADADOS DE CONTROLE (colunas com prefixo `_`) para linhagem/auditoria.
    - IDEMPOTENTE: a chave é o hash do conteúdo da fonte (`_source_hash`). Se a fonte
      não mudou e o bronze já existe, NÃO reprocessa — devolve o que já está gravado.
      Como todo metadado de dado é DETERMINÍSTICO (derivado do conteúdo, não do relógio),
      rodar 2x produz um arquivo idêntico.
    """
    os.makedirs(os.path.dirname(destino), exist_ok=True)
    brutos = _ler_fonte(fonte)
    source_hash = hashlib.sha256(brutos).hexdigest()

    # Guarda de idempotência: mesma fonte já ingerida? não reprocessa.
    if os.path.exists(meta_path) and os.path.exists(destino):
        meta_ant = json.load(open(meta_path, encoding="utf-8"))
        if meta_ant.get("source_hash") == source_hash:
            print(f"[BRONZE] Idempotente: fonte inalterada (source_hash {source_hash[:12]}...). Nada a reprocessar.")
            return pd.read_csv(destino)

    # Bronze = cópia fiel do cru + metadados de controle DETERMINÍSTICOS
    df = pd.read_csv(io.BytesIO(brutos))
    colunas_originais = list(df.columns)
    def _hash_linha(row):
        return hashlib.sha256("|".join(str(v) for v in row.values).encode("utf-8")).hexdigest()
    df["_row_hash"] = df[colunas_originais].apply(_hash_linha, axis=1)
    df["_source_hash"]  = source_hash
    df["_source_url"]   = str(fonte)
    df["_layer"]        = "bronze"
    df["_ingestion_id"] = source_hash[:12]   # determinístico (derivado do conteúdo)

    # dedup idempotente + ordem estável => saída reprodutível
    df = df.drop_duplicates("_row_hash")
    chave = "PassengerId" if "PassengerId" in colunas_originais else colunas_originais[0]
    df = df.sort_values(chave).reset_index(drop=True)
    df.to_csv(destino, index=False)

    # Sidecar de metadados do lote (linhagem/auditoria)
    meta = {
        "layer": "bronze",
        "source_url": str(fonte),
        "source_hash": source_hash,
        "ingestion_id": source_hash[:12],
        "n_linhas": int(df.shape[0]),
        "n_colunas_originais": len(colunas_originais),
        "colunas_originais": colunas_originais,
        "colunas_controle": ["_row_hash", "_source_hash", "_source_url", "_layer", "_ingestion_id"],
        "ingested_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),  # 1a ingestão (linhagem)
    }
    json.dump(meta, open(meta_path, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
    print(f"[BRONZE] Ingerido {df.shape[0]} linhas -> {destino}")
    print(f"[BRONZE] source_hash={source_hash[:12]}... | metadados em {meta_path}")
    return df


# Executa a ingestão da camada bronze
bronze = ingerir_camada_bronze()

# Próximas camadas trabalham sobre o dado CRU do bronze, descartando as colunas de
# controle (prefixo `_`) — a linhagem fica registrada no arquivo bronze + sidecar.
df = bronze.drop(columns=[c for c in bronze.columns if c.startswith("_")])
print(f"\nPASSO 1 (Bronze) - {df.shape[0]} linhas | {df.shape[1]} colunas para as próximas camadas.")


### 📋 Documentação da Camada Bronze

**Metadados de controle adicionados (colunas `_`):**

| Coluna | Significado |
|--------|-------------|
| `_row_hash` | `sha256` dos valores originais da linha — chave natural para *dedup* determinístico |
| `_source_hash` | `sha256` do conteúdo **inteiro** da fonte — chave de **idempotência** do lote |
| `_source_url` | de onde o dado veio (rastreabilidade) |
| `_layer` | camada de origem (`bronze`) |
| `_ingestion_id` | id do lote de ingestão (derivado do conteúdo, **determinístico**) |

Além disso, o *sidecar* `camada_bronze/_bronze_metadata.json` guarda a auditoria do lote
(nº de linhas/colunas, `source_hash`, `ingested_at`).

**Por que é idempotente?** Todos os metadados de dado derivam do *conteúdo* (hashes), não do
relógio; as linhas são *dedupadas* por `_row_hash` e ordenadas por `PassengerId`. Logo, a mesma
fonte sempre gera o **mesmo arquivo**. A guarda por `source_hash` ainda evita reprocessar à toa.
A célula abaixo **prova** isso rodando a ingestão de novo e comparando o `sha256` do arquivo.


In [ ]:
# Prova de IDEMPOTÊNCIA: rodar a ingestão de novo tem de manter o MESMO arquivo.
import hashlib

def _sha256_arquivo(caminho):
    return hashlib.sha256(open(caminho, "rb").read()).hexdigest()

hash_antes  = _sha256_arquivo(BRONZE_CSV)
ingerir_camada_bronze()                 # 2a execução (idempotente)
hash_depois = _sha256_arquivo(BRONZE_CSV)

assert hash_antes == hash_depois, "❌ A camada bronze NÃO é idempotente!"
print(f"✓ Idempotência confirmada: rodar 2x manteve o mesmo arquivo (sha256 {hash_depois[:12]}...).")


### 2) EXIBIR AS INFORMAÇÕES GERAIS DO DATASET 

In [5]:
print("=" * 60)
print("PASSO 2 - Informações gerais (df.info())")
print("=" * 60)
df.info()

PASSO 2 - Informações gerais (df.info())
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


### 3) EXIBIR ESTATÍSTICAS DESCRITIVAS DO DATASET

In [ ]:
print("\n" + "=" * 60)
print("PASSO 3 - Estatísticas descritivas (df.describe())")
print("=" * 60)

# Exibindo estatísticas descritivas
print(df.describe())


PASSO 3 - Estatísticas descritivas (df.describe())
       PassengerId    Survived      Pclass         Age       SibSp  \
count   891.000000  891.000000  891.000000  714.000000  891.000000   
mean    446.000000    0.383838    2.308642   29.699118    0.523008   
std     257.353842    0.486592    0.836071   14.526497    1.102743   
min       1.000000    0.000000    1.000000    0.420000    0.000000   
25%     223.500000    0.000000    2.000000   20.125000    0.000000   
50%     446.000000    0.000000    3.000000   28.000000    0.000000   
75%     668.500000    1.000000    3.000000   38.000000    1.000000   
max     891.000000    1.000000    3.000000   80.000000    8.000000   

            Parch        Fare  
count  891.000000  891.000000  
mean     0.381594   32.204208  
std      0.806057   49.693429  
min      0.000000    0.000000  
25%      0.000000    7.910400  
50%      0.000000   14.454200  
75%      0.000000   31.000000  
max      6.000000  512.329200  


### 4) VALORES AUSENTES POR COLUNA

In [7]:
print("\n" + "=" * 60)
print("PASSO 4 - Valores ausentes por coluna")
print("=" * 60)
print(df.isnull().sum())


PASSO 4 - Valores ausentes por coluna
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


### 5) PREENCHER IDADE FALTANTE UTILIZANDO MEDIANA AGRUPADA POR CLASSE E SEX

In [8]:
df["Age"] = df.groupby(["Pclass", "Sex"])["Age"].transform(
    lambda idade: idade.fillna(idade.median())
)

### 6) SE SOBRAR ALGUEM AUSENTE, PREENCHE COM A MEDIANA GERAL

In [9]:
df["Age"] = df["Age"].fillna(df["Age"].median())

print("\nPASSO 5 e 6 - Idades ausentes preenchidas.")
print(f"Valores ausentes restantes em Age: {df['Age'].isnull().sum()}")


PASSO 5 e 6 - Idades ausentes preenchidas.
Valores ausentes restantes em Age: 0


### 7) EMBARKED: PREENCHER AUSENTES COM VALOR MAIS FREQUENTE (moda)

In [10]:
porto_mais_frequente = df["Embarked"].mode()[0]
df["Embarked"] = df["Embarked"].fillna(porto_mais_frequente)

print(f"\nPASSO 7 - Embarked preenchido com o valor mais frequente: '{porto_mais_frequente}'")


PASSO 7 - Embarked preenchido com o valor mais frequente: 'S'


### 8) CABIN: CRIAR FEATURE INDICADORA (1 = TINHA CABINE, 0 = NÃO TINHA)

In [11]:
df["Has_Cabin"] = df["Cabin"].notnull().astype(int)

print("\nPASSO 8 - Nova coluna 'Has_Cabin' criada (1 = tinha cabine, 0 = não tinha).")


PASSO 8 - Nova coluna 'Has_Cabin' criada (1 = tinha cabine, 0 = não tinha).


### 9) FARE: PREENCHER AUSENTES COM A MÉDIA AGRUPADA POR CLASSE (Pclass)

In [12]:
df["Fare"] = df.groupby("Pclass")["Fare"].transform(
    lambda tarifa: tarifa.fillna(tarifa.mean())
)

print(f"\nPASSO 9 - Fare preenchido. Valores ausentes restantes: {df['Fare'].isnull().sum()}")


PASSO 9 - Fare preenchido. Valores ausentes restantes: 0


### 10) AGE: CRIAR UMA FEATURE INDICATIVA (CRIANÇA, ADOLESCENTE, ADULTO JOVEM, ADULTO, IDOSO)

In [13]:
faixas_idade = [0, 12, 18, 35, 60, 100]
nomes_idade = ["Criança", "Adolescente", "Adulto Jovem", "Adulto", "Idoso"]
df["Faixa_Idade"] = pd.cut(df["Age"], bins=faixas_idade, labels=nomes_idade)

print("\nPASSO 10 - Nova coluna 'Faixa_Idade' criada:")
print(df["Faixa_Idade"].value_counts())


PASSO 10 - Nova coluna 'Faixa_Idade' criada:
Faixa_Idade
Adulto Jovem    514
Adulto          216
Adolescente      70
Criança          69
Idoso            22
Name: count, dtype: int64


### 11) FARE: CRIAR UMA FEATURE INDICATIVA (BAIXO, MÉDIO-BAIXO, MÉDIO-ALTO, ALTO)

In [14]:
faixas_tarifa = [0, 7.91, 14.45, 31, 600]
nomes_tarifa = ["Baixo", "Médio-Baixo", "Médio-Alto", "Alto"]
df["Faixa_Tarifa"] = pd.cut(df["Fare"], bins=faixas_tarifa, labels=nomes_tarifa)

print("\nPASSO 11 - Nova coluna 'Faixa_Tarifa' criada:")
print(df["Faixa_Tarifa"].value_counts())


PASSO 11 - Nova coluna 'Faixa_Tarifa' criada:
Faixa_Tarifa
Médio-Alto     229
Alto           222
Médio-Baixo    217
Baixo          208
Name: count, dtype: int64


### RESULTADO FINAL

In [15]:
print("\n" + "=" * 60)
print("RESULTADO FINAL - primeiras linhas do dataset tratado")
print("=" * 60)
print(df[["Name", "Pclass", "Sex", "Age", "Faixa_Idade",
          "Fare", "Faixa_Tarifa", "Has_Cabin", "Embarked"]].head(10))

print("\nValores ausentes finais em todo o dataset:")
print(df.isnull().sum())



RESULTADO FINAL - primeiras linhas do dataset tratado
                                                Name  Pclass     Sex   Age  \
0                            Braund, Mr. Owen Harris       3    male  22.0   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...       1  female  38.0   
2                             Heikkinen, Miss. Laina       3  female  26.0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)       1  female  35.0   
4                           Allen, Mr. William Henry       3    male  35.0   
5                                   Moran, Mr. James       3    male  25.0   
6                            McCarthy, Mr. Timothy J       1    male  54.0   
7                     Palsson, Master. Gosta Leonard       3    male   2.0   
8  Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)       3  female  27.0   
9                Nasser, Mrs. Nicholas (Adele Achem)       2  female  14.0   

    Faixa_Idade     Fare Faixa_Tarifa  Has_Cabin Embarked  
0  Adulto Jovem   7.2500  

### SALVA TUDO EM CSV

In [ ]:
import os
# Camada OURO (gold): dataset tratado e pronto para análise/modelo.
os.makedirs("camada_gold", exist_ok=True)
df.to_csv("camada_gold/titanic_tratado.csv", index=False)
print("\nCamada gold salva em 'camada_gold/titanic_tratado.csv'.")